# STL-10 Reconstruction: Plain Autoencoder vs. Small U-Net

Direct comparison of the two reconstruction architectures already
built for STL-10 in this repo, trained with **identical**
hyperparameters (same seed, epochs, batch size, learning rate, same
train/test split, same MSE reconstruction loss, no noise added -- pure
reconstruction `L(x) = ||x - model(x)||^2`), so any quality gap
reflects the architecture difference rather than the training setup.

| | `STL10ConvAutoencoder` (`code/stl10_conv_autoencoder.py`) | `SmallUNet` (`code/stl10_unet.py`) |
|---|---|---|
| bottleneck | FC layer, flattened to `latent_dim=128` numbers | convolutional, `[B,256,12,12]` feature map, no FC |
| skip connections | none -- every pixel must be reconstructed from the 128-number vector | 3, one per resolution level (96/48/24) |

Core code lives in `code/compare_stl10_autoencoder_vs_unet.py`
(version-controlled), which imports both models directly from their
own files rather than redefining them. This notebook is a thin
orchestration layer: GPU check, repo pull, download real STL-10
(cached in Drive), train both models back-to-back, print a
side-by-side metrics table (parameter count, test MSE, test PSNR, test
SSIM), plot both loss curves on one figure, and show original /
autoencoder-reconstruction / U-Net-reconstruction triplets as the
required results-visualization step.

**What is controlled vs. not**: random seed, epochs, batch size,
learning rate, optimizer, loss function, and the dataset/split are all
identical for both models (see the module docstring in the `.py`
file). The exact mini-batch order is not bit-identical between the two
training runs, since each model's training loop draws independently
from the shared DataLoader's shuffling RNG -- standard for this kind
of comparison, but stated explicitly per this project's research-
integrity convention.

## 1. GPU check

In [ ]:
import torch

# Never hard-code CUDA: fall back to CPU if no GPU is available.
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"device = {device}")
if device.type == "cuda":
    print(torch.cuda.get_device_name(0))


## 2. Clone/pull repository

Model + comparison code lives in `code/stl10_conv_autoencoder.py`,
`code/stl10_unet.py`, and `code/compare_stl10_autoencoder_vs_unet.py`,
not duplicated in this notebook. Replace `REPO_URL` with this
repository's URL.

In [ ]:
import os

REPO_URL = "https://github.com/RoeiZ/from_zero_to_hero.git"
REPO_DIR = "/content/from_zero_to_hero"

if not os.path.isdir(REPO_DIR):
    !git clone $REPO_URL $REPO_DIR
else:
    !cd $REPO_DIR && git pull

import sys
sys.path.insert(0, os.path.join(REPO_DIR, "code"))


## 3. Mount Drive (persistent STL-10 cache)

Same dataset loader (`get_stl10_dataloaders`) and same persistent
Drive path convention as `stl10_conv_autoencoder.ipynb` /
`stl10_unet.ipynb`, so STL-10 is downloaded once and reused across
sessions.

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

# Persistent cache: downloaded once into Drive, reused on every
# future Colab session (local /content storage is ephemeral).
DATA_ROOT = "/content/drive/MyDrive/thesis_data/stl10"


## 4. Train both models with identical hyperparameters

`run_comparison` builds fresh train/test DataLoaders once (shared by
both models), then trains `STL10ConvAutoencoder` and `SmallUNet` from
scratch, back-to-back, with the same seed/epochs/batch size/learning
rate. Real GPU training -- Colab only, same epoch count as the
standalone autoencoder/U-Net notebooks (15) for a fair comparison.

In [ ]:
from compare_stl10_autoencoder_vs_unet import run_comparison, print_comparison_summary

BATCH_SIZE = 64
N_EPOCHS = 15
LR = 1e-3
LATENT_DIM = 128  # autoencoder bottleneck size
SEED = 0

results = run_comparison(
    DATA_ROOT,
    batch_size=BATCH_SIZE,
    epochs=N_EPOCHS,
    lr=LR,
    latent_dim=LATENT_DIM,
    device=device,
    seed=SEED,
)

print_comparison_summary(results["summary"])


## 5. Loss curves: autoencoder vs. U-Net

Train and test MSE per epoch, both models on the same axes. A lower,
faster-dropping U-Net curve would indicate the skip connections let it
fit the reconstruction task more easily than the FC bottleneck.

In [ ]:
from compare_stl10_autoencoder_vs_unet import plot_loss_comparison

plot_loss_comparison(results["autoencoder_history"], results["unet_history"])


## 6. Qualitative + quantitative check: original vs. both reconstructions

Per-project convention: always look at concrete examples, not only
scalar metrics. Shows 5 real test images as original (row 1) /
autoencoder reconstruction (row 2) / U-Net reconstruction (row 3),
with per-image MSE, PSNR, and SSIM (each vs. the original) under every
reconstruction.

In [ ]:
from compare_stl10_autoencoder_vs_unet import visualize_three_way_comparison

sample_images, _sample_labels = next(iter(results["test_loader"]))
sample_images = sample_images.to(device)

visualize_three_way_comparison(results["autoencoder"], results["unet"], sample_images, num_examples=5)


## 7. Final summary table

Parameter count, test-set MSE, PSNR, and SSIM for both models,
repeated here for convenience after training.

In [ ]:
print_comparison_summary(results["summary"])
